In [ ]:
import os
os.environ["GROQ_API_KEY"] = " " # <<< REPLACE THIS WITH YOUR ACTUAL GROQ API KEY
import gradio as gr
from explain import explain_topic
from mcq import generate_mcqs
from writting import writing_helper
from code_helper import generate_code
from quiz import generate_quiz, parse_quiz, evaluate_quiz
import time


In [2]:
def toggle_highschool(level):
    """
    Show Grade & Board if 'High School' is selected,
    hide them if 'Graduation' is selected.
    """
    if level == "High School":
        return gr.update(visible=True), gr.update(visible=True)
    else:  # Graduation
        return gr.update(visible=False), gr.update(visible=False)


In [3]:
quiz_data = []
start_time = None

In [4]:
def create_quiz(topics, grade):
    global quiz_data, start_time

    for _ in range(3):  # retry 3 times
        raw = generate_quiz(topics, grade, num_questions=20)
        quiz_data = parse_quiz(raw)

        if len(quiz_data) >= 11:
            break

    start_time = time.time()

    outputs = []

    for i in range(11):
        if i < len(quiz_data):
            q = quiz_data[i]
            outputs.append(f"### {q['question']}")
            outputs.append(gr.update(choices=q["options"], value=None))
        else:
            outputs.append(f"### Q{i+1}. Not available")
            outputs.append(gr.update(choices=[], value=None))

    return outputs


def submit_quiz(*answers):
    global quiz_data, start_time

    if not quiz_data:
        return "⚠️ Generate quiz first"

    n = min(len(quiz_data), len(answers))

    for i in range(n):
        if answers[i] is None:
            return "⚠️ Please answer all questions"

    end_time = time.time()
    total_time = end_time - start_time

    score = 0

    for i in range(n):
        user = answers[i]
        correct = quiz_data[i]["answer"]

        # ✅ Only compare first character
        user_option = user.strip().lower()[0]
        correct_option = correct.strip().lower()[0]

        if user_option == correct_option:
            score += 1

    return f"✅ Score: {score}/{n} | ⏱ Time: {round(total_time,2)} sec"

In [5]:
with gr.Blocks() as demo:
    gr.Markdown("# 📘 AI Tutor - Student Helper")

    # ---------------- Explanation Tab ----------------
    with gr.Tab("Explanation"):
        topic_exp = gr.Textbox(label="Enter Topic", placeholder="e.g., Science, Math, History")
        level_exp = gr.Dropdown(choices=["High School", "Graduation"], label="Student Level", value="High School")
        grade_exp = gr.Dropdown(choices=[f"{i}th" for i in range(5, 13)], label="Grade (5th-12th)", visible=True)
        board_exp = gr.Dropdown(choices=["CBSE", "ICSE", "SSC"], label="Education Board", visible=True)

        level_exp.change(fn=toggle_highschool, inputs=[level_exp], outputs=[grade_exp, board_exp])

        output_exp = gr.Markdown(label="Explanation")
        btn_exp = gr.Button("Generate Explanation")
        btn_exp.click(fn=explain_topic, inputs=[topic_exp, level_exp, grade_exp, board_exp], outputs=output_exp)

    # ---------------- MCQ Generator Tab ----------------
    with gr.Tab("MCQ Generator"):
        topic_mcq = gr.Textbox(label="Enter Topic for MCQs")
        level_mcq = gr.Dropdown(choices=["High School", "Graduation"], label="Student Level", value="High School")
        grade_mcq = gr.Dropdown(choices=[f"{i}th" for i in range(5, 13)], label="Grade (5th-12th)", visible=True)
        board_mcq = gr.Dropdown(choices=["CBSE", "ICSE", "SSC"], label="Education Board", visible=True)

        level_mcq.change(fn=toggle_highschool, inputs=[level_mcq], outputs=[grade_mcq, board_mcq])

        n_mcq = gr.Slider(minimum=5, maximum=15, value=5, step=1, label="Number of MCQs")
        output_mcq = gr.Markdown(label="Generated MCQs")
        btn_mcq = gr.Button("Generate MCQs")
        btn_mcq.click(fn=generate_mcqs, inputs=[topic_mcq, level_mcq, n_mcq, grade_mcq, board_mcq], outputs=output_mcq)

    # ---------------- Writing Assistant Tab ----------------
    with gr.Tab("Writing Assistant"):
        topic_write = gr.Textbox(label="Enter Topic")
        writing_type = gr.Dropdown(choices=["Essay", "Letter", "Paragraph", "Report"], label="Select Writing Type")
        language = gr.Dropdown(choices=["English", "Hindi", "Marathi"], label="Select Language")
        grade_write = gr.Dropdown(choices=[f"{i}th" for i in range(5, 13)], label="Grade")
        board_write = gr.Dropdown(choices=["CBSE", "ICSE", "SSC"], label="Education Board")

        output_write = gr.Textbox(label="Generated Output", lines=12)
        btn_write = gr.Button("Generate Writing")
        btn_write.click(fn=writing_helper, inputs=[topic_write, writing_type, language, grade_write, board_write], outputs=output_write)

    # ---------------- Coding Assistant Tab ----------------
    with gr.Tab("Coding Assistant"):
        gr.Markdown("Competitive Programming Helper")


        question_text = gr.Textbox(
            label="Paste Full Question",
            lines=5,
            placeholder="Paste the complete problem statement here..."
        )

        language_code = gr.Dropdown(
            choices=["Python", "Java", "C++", "JavaScript"],
            label="Select Programming Language",
            value="Python"
        )

        output_code = gr.Markdown(label="Generated Solution")

        btn_generate_code = gr.Button("Generate Solution")

        btn_generate_code.click(
            fn=generate_code,
            inputs=[question_text, language_code],
            outputs=output_code
        )
    # ---------------- Quiz Tab ----------------
    with gr.Tab("Quiz"):
        gr.Markdown("## 🎯 Quiz Competition")

        grade = gr.Dropdown(
            choices=["5th", "6th", "7th", "8th", "9th", "10th"],
            label="Select Grade"
        )

        topics = gr.Dropdown(
            choices=["Science", "Social Studies", "Maths", "Computer", "English Grammar"],
            multiselect=True,
            label="Select Topics"
        )

        generate_btn = gr.Button("Generate Quiz")

        # 👇 Create question blocks FIRST
        q_blocks = []
        for i in range(10):
            with gr.Group():
                q_text = gr.Markdown(f"### Q{i+1}")
                q_radio = gr.Radio(
                    choices=["a", "b", "c", "d"],  # temporary default
                    label="Select Answer"
                )
            q_blocks.append((q_text, q_radio))

        submit_btn = gr.Button("Submit Quiz")
        result = gr.Textbox(label="Result")

        # ✅ 👉 ADD HERE (after q_blocks is defined)
        generate_btn.click(
            create_quiz,
            inputs=[topics, grade],
            outputs=[item for pair in q_blocks for item in pair]
        )

        submit_btn.click(
            submit_quiz,
            inputs=[radio for _, radio in q_blocks],
            outputs=result
        )


In [6]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff96350703bc3c3bb0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
